In [ ]:
!pip install lxml[html_clean]
!apt-get update
!apt-get install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium newspaper3k nltk bs4 requests scikit-learn gensim

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('popular')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to /root/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to /root/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to /root/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package movie_reviews is already up-to-date!
[nltk_data]    | Downloading 

True

In [ ]:
!pip install xlsxwriter

In [ ]:
# Import các thư viện cần thiết cho quá trình scraping, xử lý dữ liệu, và xuất file
import os
import re
import time
from datetime import datetime
import random
import json
import requests
import tempfile
import atexit
from tqdm import tqdm
import pandas as pd
from bs4 import BeautifulSoup
from newspaper import Article
from selenium import webdriver  # Thư viện để tự động hóa trình duyệt, hỗ trợ scraping động
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException
from selenium.webdriver.common.keys import Keys
from google.colab import files
from urllib.parse import urljoin

# ***TIN NHANH CHUNG KHOAN***

In [ ]:
import requests, re, unicodedata
from openpyxl import load_workbook
from openpyxl.styles import Font

# 1. Bản đồ mã → tên tiếng Việt & Anh
company_map = {
    "mã VNM": ("CTCP Sữa Việt Nam", "Công ty cổ phần Sữa Việt Nam"),
    "mã ANV": ("Công ty cổ phần Nam Việt", "CTCP Nam Việt"),
    "mã BMP": ("Công ty cổ phần Nhựa Bình Minh", "CTCP Nhựa Bình Minh"),
    "mã BSI": ("Công ty Cổ phần Chứng khoán BIDV", "CTCP Chứng khoán BIDV"),
    "mã CMG": ("CMC Corp", "CTCP Tập đoàn Công nghệ CMC "),
    "mã CTD": ("Công ty Cổ phần Xây dựng Coteccons", "CTCP xây dựng Coteccons","Mã chứng khoán: CTD - HOSE"),
    "mã DBC": ("Công ty Cổ phần Tập đoàn Dabaco Việt Nam", "CTCP Tập đoàn Dabaco Việt Nam","DBC – sàn HOSE","Mã chứng khoán: DBC - HOSE"),
    "mã DIG": ("Tổng công ty cổ phần Đầu tư Phát triển Xây dựng", "Tổng CTCP Đầu tư phát triển xây dựng","DIC Corp"),
    "mã FPT": ("FPT Corporation", "Tập đoàn FPT","CTCP FPT"),
    "mã GAS": ("Tổng công ty Khí Việt Nam", "PV GAS","Tổng công ty Khí Việt Nam – CTCP","Tổng CTCP Gas Petrolimex"),
    "mã GMD": ("Công ty cổ phần Gemadept", "CTCP Gemadept"),
    "mã HCM": ("CTCP Chứng khoán TP. HCM", "Công ty cổ phần Chứng khoán TP. HCM","Chứng khoán HSC"),
    "mã HDG": ("CTCP Tập đoàn Hà Đô", "Công ty Cổ phần Tập đoàn Hà Đô"),
    "mã HPG": ("CTCP Tập đoàn Hòa Phát", "Công ty Cổ phần Tập đoàn Hòa Phát"),
    "mã HSG": ("Tập đoàn Hoa Sen", "CTCP Tập đoàn Hoa Sen"),
    "mã IMP": ("Công ty Cổ phần Dược phẩm Imexpharm", "CTCP Dược phẩm Imexpharm"),
    "mã KBC": ("Tổng CTCP Phát triển Đô thị Kinh Bắc", "Phát triển Đô thị Kinh Bắc"),
    "mã KDH": ("CTCP Khang Điền", "Khang Điền"),
    "mã MBB": ("Ngân hàng TMCP Quân đội", "Ngân hàng thương mại cổ phần Quân Đội","MBBank"),
    "mã NKG": ("CTCP Thép Nam Kim", "Nam Kim"),
    "mã PDR": ("CTCP Phát triển Bất động sản Phát Đạt", "Công ty cổ phần Phát triển Bất động sản Phát Đạt"),
    "mã PNJ": ("CTCP Vàng bạc đá quý Phú Nhuận", "Công ty cổ phần Vàng bạc đá quý Phú Nhuận"),
    "mã PTB": ("Công ty Cổ phần Phú Tài", "CTCP Phú Tài"),
    "mã REE": ("Cơ điện lạnh (REE)", "CTCP Cơ điện lạnh","Công ty cổ phần Cơ điện lạnh"),
    "mã SSI": ("CTCP Chứng khoán SSI", "Công ty cổ phần Chứng khoán SSI"),
    "mã VCB": ("Ngân hàng TMCP Ngoại thương Việt Nam", "Ngân hàng Thương mại cổ phần Ngoại thương Việt Nam"),
    "mã VIX": ("CTCP Chứng khoán VIX", "Công ty cổ phần Chứng khoán VIX"),
    "mã VND": ("VNDirect", "CTCP Chứng khoán VNDirect"),
    "mã VSC": ("Viconship", "CTCP Container Việt Nam","Công ty Cổ phần Container Việt Nam"),
    "mã TLG": ("Tập đoàn Thiên Long",)
}

# 2. Tạo từ khóa: mã + các tên tiếng Việt
ticker_keywords = {
    ticker: [ticker.lower()] + [name.lower() for name in names]
    for ticker, names in company_map.items()
}

# 3. Hàm làm sạch văn bản
def clean_text(txt):
    if not txt:
        return ""
    reps = {'\u200b':'','\u00a0':' ', '\u2013':'-','\u2014':'--',
            '\u2018':"'",'\u2019':"'", '\u201c':'"','\u201d':'"',
            '\u2026':'...'}
    for a, b in reps.items():
        txt = txt.replace(a, b)
    return re.sub(r'[\x00-\x1f\x7f-\x9f]', '', txt).strip()

# 4. Lấy link từ sitemap theo ngày
def get_sitemap_urls(url, from_dt, to_dt):
    r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"})
    soup = BeautifulSoup(r.text, 'xml')
    out = []
    for u in soup.find_all('url'):
        lm = u.find('lastmod')
        if not lm:
            continue
        try:
            dt = datetime.fromisoformat(lm.text).replace(tzinfo=None)
        except:
            continue
        if from_dt <= dt < to_dt:
            loc = u.find('loc').text.strip()
            out.append((loc, dt.strftime("%d/%m/%Y %H:%M")))
    return out

# 5. Crawl nội dung và xác định ticker
def fetch(link):
    r = requests.get(link, headers={"User-Agent":"Mozilla/5.0"}, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")
    # Lấy tiêu đề
    title_tag = soup.select_one("h1.article__header.cms-title")
    title = clean_text(title_tag.get_text(strip=True)) if title_tag else ""
    # Lấy tóm tắt và nội dung để kiểm tra mã
    sumpt = soup.select_one("div.article__sapo.cms-desc")
    body_div = soup.select_one("div.article__body.cms-body")
    summary = clean_text(sumpt.get_text(strip=True)) if sumpt else ""
    if body_div and body_div.find(["iframe","video","embed"]):
        return None
    paragraphs = body_div.find_all("p") if body_div else []
    body = "\n".join(clean_text(p.get_text(strip=True)) for p in paragraphs)
    if len(summary) + len(body) < 200:
        return None
    content = summary  # Chỉ lấy tóm tắt
    full_lower = (summary + " " + body).lower()  # Kiểm tra mã trong cả tóm tắt và nội dung

    found = {t: kws for t, kws in ticker_keywords.items()
             if any(k in full_lower for k in kws)}
    if not found:
        return None

    keywords = sum(found.values(), [])
    return {"title": title, "content": content, "tickers": list(found.keys()), "keywords": keywords}

# 6. In đậm các từ khóa trong Content
def apply_bold(ws, row, content, kws):
    cell = ws.cell(row=row, column=7)
    if not kws:
        cell.value = content
        return
    patt = "|".join(re.escape(k) for k in kws)
    out, last = [], 0
    for m in re.finditer(patt, content, re.IGNORECASE):
        out.append(content[last:m.start()])
        out.append(f"**{content[m.start():m.end()]}**")
        last = m.end()
    out.append(content[last:])
    cell.value = "".join(out)

# 7. Main
def main():
    base = "https://www.tinnhanhchungkhoan.vn/sitemaps/"
    maps = ["news-2025-6.xml", "news-2025-1.xml"]
    from_dt = datetime(2025,1,1)
    to_dt   = datetime(2025,6,30)
    rows = []

    for sm in maps:
        for link, dt in get_sitemap_urls(base+sm, from_dt, to_dt):
            res = fetch(link)
            if not res:
                continue
            tickers = res["tickers"]
            kws = res["keywords"]
            rows.append({
                "Date": dt,
                "Link": link,
                "Tickers": ", ".join(tickers),
                "Tiêu Đề": res["title"],
                "Content": res["content"],
                "Keywords": ", ".join(kws)
            })
            print(f"✅ {dt} | {link} -> {rows[-1]['Tickers']} | {rows[-1]['Tiêu Đề']}")

    if not rows:
        print("❌ Không có bài viết phù hợp.")
        return

    df = pd.DataFrame(rows)
    fn = "multi_tickers_combined.xlsx"
    df.to_excel(fn, index=False, engine="openpyxl")

    wb = load_workbook(fn)
    ws = wb.active
    widths = [15, 50, 20, 40, 60, 100, 30]  # Điều chỉnh chiều rộng cho cột Tiêu Đề
    for col, w in zip("ABCDEFG", widths):
        ws.column_dimensions[col].width = w

    for idx, r in enumerate(rows, start=2):
        apply_bold(ws, idx, r["Content"], r["Keywords"].split(", "))

    for cell in ws[1]: cell.font = Font(bold=True)
    wb.save(fn)
    print(f"🎉 Đã lưu {len(rows)} dòng vào '{fn}'")

if __name__ == "__main__":
    main()

✅ 29/06/2025 10:12 | https://www.tinnhanhchungkhoan.vn/ha-tang-tai-san-so-tai-viet-nam-tay-choi-lon-bat-dau-hanh-dong-post372062.html -> mã SSI | Hạ tầng tài sản số tại Việt Nam: "Tay chơi lớn" bắt đầu hành động
✅ 28/06/2025 10:27 | https://www.tinnhanhchungkhoan.vn/hai-an-hah-len-tieng-ve-moi-quan-he-hop-tac-voi-viconship-vsc-sau-khi-nhom-co-dong-nang-so-huu-len-1531-post372019.html -> mã VSC | Hải An (HAH) lên tiếng về mối quan hệ hợp tác với Viconship (VSC) sau khi nhóm cổ đông nâng sở hữu lên 15,31%
✅ 28/06/2025 10:24 | https://www.tinnhanhchungkhoan.vn/gemadept-gmd-sap-chi-hon-84038-ty-dong-de-tra-co-tuc-nam-2024-post372020.html -> mã GMD | Gemadept (GMD) sắp chi hơn 840,38 tỷ đồng để trả cổ tức năm 2024
✅ 27/06/2025 18:02 | https://www.tinnhanhchungkhoan.vn/su-kien-chung-khoan-dang-chu-y-ngay-286-post372012.html -> mã GMD | Sự kiện chứng khoán đáng chú ý ngày 28/6
✅ 27/06/2025 15:00 | https://www.tinnhanhchungkhoan.vn/dhcd-phat-dat-pdr-chuyen-nhuong-80-du-an-thuan-an-1-cho-doi-ta

KeyboardInterrupt: 

# ***VIETSTOCK***

In [ ]:
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")  # nếu muốn thấy trình duyệt, comment dòng này
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("--disable-extensions")
    # Chặn ảnh để nhanh/nhẹ
    chrome_options.add_experimental_option("prefs", {
        "profile.managed_default_content_settings.images": 2
    })

    driver = webdriver.Chrome(options=chrome_options)
    # Ẩn một số dấu hiệu automation
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
            Object.defineProperty(navigator, 'languages', { get: () => ['vi-VN','vi'] });
            Object.defineProperty(navigator, 'plugins', { get: () => [1,2,3] });
        """
    })
    return driver

# =========================
# HỖ TRỢ: tìm input & set value + bắn event
# =========================
def _find_clickable_css(driver, selectors, timeout=30):
    """Tìm element theo nhiều CSS, ưu tiên cái hiện & click được."""
    end = time.time() + timeout
    last_err = None
    while time.time() < end:
        for sel in selectors:
            try:
                el = driver.find_element(By.CSS_SELECTOR, sel)
                if el.is_displayed():
                    try:
                        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, sel)))
                    except Exception:
                        pass
                    return sel, el
            except Exception as e:
                last_err = e
        time.sleep(1)
    raise last_err or TimeoutException("Không tìm thấy input ngày")

def _robust_set_input(driver, el, value):
    """Set input: gỡ readonly, focus, Ctrl+A, delete, send_keys, rồi bắn events bằng JS."""
    driver.execute_script("arguments[0].removeAttribute('readonly');arguments[0].removeAttribute('disabled');", el)
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    try:
        el.click()
    except Exception:
        pass
    try:
        el.send_keys(Keys.CONTROL, 'a')
    except Exception:
        pass
    try:
        el.send_keys(Keys.DELETE)
    except Exception:
        pass
    try:
        el.send_keys(value)
    except Exception:
        # fallback set bằng JS
        driver.execute_script("arguments[0].value = arguments[1];", el, value)

    # bắn sự kiện để UI lib/datepicker bắt thay đổi
    driver.execute_script("""
        const el = arguments[0];
        el.dispatchEvent(new Event('input',  {bubbles:true}));
        el.dispatchEvent(new Event('change', {bubbles:true}));
        try {
          if (window.jQuery) {
            const $el = jQuery(el);
            if ($el.data && ($el.data('datepicker') || $el.data('DateTimePicker'))) {
              try { $el.datepicker && $el.datepicker('setDate', el.value); } catch(e) {}
              $el.trigger && $el.trigger('changeDate');
            }
          }
        } catch(e) {}
    """, el)

# =========================
# SET DATE FILTER (ĐÃ SỬA)
# =========================
def set_date_filter(driver, from_date, to_date):
    """
    - Tìm đúng input: #txtFromDate/#txtToDate bên trong .input-group.date
    - Gỡ readonly, set bằng send_keys + JS
    - Click 'Tìm kiếm'
    - Chờ #latest-news thay đổi nội dung
    """
    try:
        WebDriverWait(driver, 35).until(EC.presence_of_element_located((By.ID, "latest-news")))

        # Tìm input theo nhiều khả năng (theo DOM bạn gửi)
        from_css_list = [
            "#txtFromDate input.form-control",
            "#txtFromDate input",
            "input[name='FromDate']",
            "input#FromDate",
        ]
        to_css_list = [
            "#txtToDate input.form-control",
            "#txtToDate input",
            "input[name='ToDate']",
            "input#ToDate",
        ]

        from_sel, from_el = _find_clickable_css(driver, from_css_list, timeout=20)
        to_sel,   to_el   = _find_clickable_css(driver, to_css_list,   timeout=20)

        # HTML cũ để chờ thay đổi
        try:
            old_html = driver.find_element(By.ID, "latest-news").get_attribute("innerHTML")
        except Exception:
            old_html = None

        # Set giá trị vào 2 input
        _robust_set_input(driver, from_el, from_date)
        _robust_set_input(driver, to_el,   to_date)

        # Debug: in giá trị thực tế trong DOM sau khi set
        vals = driver.execute_script("""
            return {
              from_input: document.querySelector(arguments[0])?.value || null,
              to_input:   document.querySelector(arguments[1])?.value || null
            };
        """, from_sel, to_sel)
        print("Date values (after set):", vals)

        # Click 'Tìm kiếm'
        btn = WebDriverWait(driver, 30).until(EC.element_to_be_clickable((By.ID, "btn-news-filter")))
        try:
            btn.click()
        except Exception:
            driver.execute_script("arguments[0].click();", btn)

        # Chờ nội dung đổi
        if old_html is not None:
            WebDriverWait(driver, 35).until(
                lambda d: d.find_element(By.ID, "latest-news").get_attribute("innerHTML") != old_html
            )
        else:
            WebDriverWait(driver, 25).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "#latest-news tr"))
            )

        # In thử 3–5 ngày đầu để kiểm tra
        try:
            html = driver.find_element(By.ID, "latest-news").get_attribute("innerHTML")
            soup = BeautifulSoup(html, "html.parser")
            sample_dates = [td.get_text(strip=True) for td in soup.select("td.col-date")[:5]]
            print("Sample dates:", sample_dates)
        except Exception:
            pass

        return True

    except Exception as e:
        # In traceback cho dễ chẩn đoán
        try:
            import traceback
            print("Lỗi thiết lập bộ lọc chi tiết:\n", traceback.format_exc())
        except Exception:
            pass
        print(" Lỗi thiết lập bộ lọc:", getattr(e, "msg", str(e)))
        return False

# =========================
# GET TOTAL PAGES (giữ nguyên)
# =========================
def get_total_pages(driver):
    try:
        pagination = driver.find_element(By.ID, "latest-paging")
        page_links = pagination.find_elements(By.CSS_SELECTOR, "li a[page]")

        if page_links:
            pages = [int(link.get_attribute("page")) for link in page_links if link.get_attribute("page").isdigit()]
            return max(pages) if pages else 1
        return 1
    except Exception as e:
        print(f" Không thể lấy số trang: {e}")
        return 1

# =========================
# EXTRACT NEWS (giữ nguyên)
# =========================
def extract_news_from_page(driver, symbol):
    news_list = []
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "latest-news"))
        )

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        news_table = soup.find('div', id='latest-news')

        if news_table:
            rows = news_table.find_all('tr')

            for row in rows:
                try:
                    date_cell = row.find('td', class_='col-date')
                    if not date_cell:
                        continue

                    date_str = date_cell.text.strip()

                    link_cell = row.find('a', class_='news-link')
                    if not link_cell:
                        continue

                    link = link_cell.get('href', '')
                    if link.startswith('//'):
                        link = 'https:' + link
                    elif link.startswith('/'):
                        link = 'https://vietstock.vn' + link

                    title_element = link_cell.find(string=True, recursive=False)
                    if not title_element:
                        title = link_cell.get_text(strip=True)
                    else:
                        title = title_element.strip()

                    if date_str and title and link:
                        news_list.append({
                            'Date': date_str,
                            'Symbol': symbol,
                            'Link': link,
                            'Title': title
                        })
                except Exception:
                    continue
    except Exception as e:
        print(f" Lỗi trích xuất tin tức: {e}")

    return news_list

# =========================
# AJAX PAGING (ĐÃ SỬA: dùng FromDate/ToDate)
# =========================
def crawl_page_with_ajax(driver, symbol, page_num, from_date, to_date):
    try:
        ajax_script = f"""
        (function() {{
          const container = document.querySelector('#latest-news');
          if (!container) return 'no_container';
          const oldHTML = container.innerHTML;

          const params = new URLSearchParams();
          params.append('view', '1');
          params.append('code', '{symbol}');
          params.append('type', '1');
          // Sử dụng PascalCase như nhiều payload trên site
          params.append('FromDate', '{from_date}');
          params.append('ToDate', '{to_date}');
          params.append('channelID', '1');
          params.append('page', '{page_num}');
          params.append('pageSize', '20');

          return fetch('/View/PagingNewsContent', {{
            method: 'POST',
            headers: {{ 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8' }},
            body: params
          }}).then(r => r.text()).then(html => {{
            container.innerHTML = html;
            return 'ok';
          }});
        }})();
        """
        old_html = driver.find_element(By.ID, "latest-news").get_attribute("innerHTML")
        driver.execute_script(ajax_script)

        # Đợi nội dung đổi
        WebDriverWait(driver, 45).until(
            lambda d: d.find_element(By.ID, "latest-news").get_attribute("innerHTML") != old_html
        )
        return True
    except Exception as e:
        print(f"Lỗi AJAX trang {page_num}: {e}")
        return False

# =========================
# NAVIGATE PAGING (giữ nguyên)
# =========================
def navigate_to_page(driver, page_num):
    try:
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)

        page_link = WebDriverWait(driver, 45).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"#latest-paging a[page='{page_num}']"))
        )

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", page_link)
        old_html = driver.find_element(By.ID, "latest-news").get_attribute("innerHTML")
        time.sleep(1.5)

        try:
            page_link.click()
        except Exception:
            driver.execute_script("arguments[0].click();", page_link)

        WebDriverWait(driver, 45).until(
            lambda d: d.find_element(By.ID, "latest-news").get_attribute("innerHTML") != old_html
        )
        return True
    except Exception as e:
        print(f" Không thể chuyển trang {page_num}: {e}")
        return False

# =========================
# CRAWLER (giữ nguyên luồng, dùng hàm đã sửa)
# =========================
def crawl_vietstock_news(symbols, from_date="01/01/2014", to_date="31/12/2024"):
    all_news = []
    driver = setup_driver()

    try:
        for symbol in symbols:
            print(f"\nCrawling: {symbol}")

            url = f"https://finance.vietstock.vn/{symbol}/tin-moi-nhat.htm"
            driver.get(url)
            WebDriverWait(driver, 45).until(
                EC.presence_of_element_located((By.ID, "latest-news"))
            )

            if not set_date_filter(driver, from_date, to_date):
                print(f" Không thể thiết lập bộ lọc cho {symbol}")
                continue

            total_pages = get_total_pages(driver)
            print(f"📄 Tổng số trang: {total_pages}")

            symbol_news_count = 0

            for page in range(1, total_pages + 1):
                if page > 1:
                    if not crawl_page_with_ajax(driver, symbol, page, from_date, to_date):
                        if not navigate_to_page(driver, page):
                            continue

                news_data = extract_news_from_page(driver, symbol)
                page_count = len(news_data)
                symbol_news_count += page_count

                print(f"   Trang {page}: {page_count} bài")

                all_news.extend(news_data)
                time.sleep(1)  # nghỉ nhẹ tránh bị chặn

            print(f" {symbol}: {symbol_news_count} bài tổng cộng")

    except Exception as e:
        print(f"Lỗi crawl: {e}")
    finally:
        driver.quit()

    return all_news

# =========================
# SAVE EXCEL (giữ nguyên)
# =========================
def save_to_excel(news_data, filename="vietstock_news.xlsx"):
    if not news_data:
        print(" Không có dữ liệu để lưu")
        return

    try:
        df = pd.DataFrame(news_data)
        df['Date_parsed'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
        df = df.sort_values(['Symbol', 'Date_parsed'], ascending=[True, False])
        df = df.drop('Date_parsed', axis=1)

        df.to_excel(filename, index=False, engine='openpyxl')
        print(f"\nĐã lưu {len(df)} bài báo → {filename}")

        print("\nThống kê:")
        for symbol in df['Symbol'].unique():
            count = len(df[df['Symbol'] == symbol])
            print(f"   {symbol}: {count} bài")
    except Exception as e:
        print(f"Lỗi lưu file: {e}")

# =========================
# MAIN (giữ nguyên)
# =========================
def main():
    symbols = ['BMP', 'BSI']
    from_date = "01/01/2014"
    to_date = "31/12/2024"

    print("VIETSTOCK NEWS CRAWLER")
    print(f"Mã: {', '.join(symbols)}")
    print(f"Thời gian: {from_date} → {to_date}")
    print()

    news_data = crawl_vietstock_news(symbols, from_date, to_date)

    if news_data:
        save_to_excel(news_data, "vietstock_news_dbc_ctd.xlsx")
        print(f" Hoàn thành! Tổng cộng: {len(news_data)} bài báo")
    else:
        print("Không có dữ liệu")

if __name__ == "__main__":
    main()

VIETSTOCK NEWS CRAWLER
Mã: BMP, BSI
Thời gian: 01/01/2014 → 31/12/2024


Crawling: BMP
Date values (after set): {'from_input': '01/01/2014', 'to_input': '31/12/2024'}
Sample dates: ['23/12/2024', '04/12/2024', '14/11/2024', '22/10/2024', '18/10/2024']
📄 Tổng số trang: 29
   Trang 1: 20 bài
   Trang 2: 20 bài
   Trang 3: 20 bài
   Trang 4: 20 bài
   Trang 5: 20 bài
   Trang 6: 20 bài
   Trang 7: 0 bài
   Trang 8: 20 bài
   Trang 9: 20 bài
   Trang 10: 20 bài
   Trang 11: 20 bài
   Trang 12: 20 bài
   Trang 13: 20 bài
   Trang 14: 20 bài
   Trang 15: 20 bài
   Trang 16: 20 bài
   Trang 17: 20 bài
   Trang 18: 20 bài
   Trang 19: 20 bài
   Trang 20: 20 bài
   Trang 21: 20 bài
   Trang 22: 20 bài
   Trang 23: 20 bài
   Trang 24: 20 bài
   Trang 25: 20 bài
   Trang 26: 20 bài
   Trang 27: 20 bài
   Trang 28: 20 bài
   Trang 29: 15 bài
 BMP: 555 bài tổng cộng

Crawling: BSI
Date values (after set): {'from_input': '01/01/2014', 'to_input': '31/12/2024'}
Sample dates: ['30/12/2024', '30/12/202

In [ ]:
# Tải lại excel để để crawl nội dung pHead
from google.colab import files
uploaded = files.upload()
excel_path = list(uploaded.keys())[0]
df = pd.read_excel(excel_path)
df


Saving vietstock_news_bmp_bsi.xlsx to vietstock_news_bmp_bsi.xlsx


,Date,Symbol,Link,Title
0,23/12/2024,BMP,https://vietstock.vn/2024/12/phan-tich-ky-thua...,Phân tích kỹ thuật phiên chiều 23/12: Tình hìn...
1,04/12/2024,BMP,https://vietstock.vn/2024/12/phan-tich-ky-thua...,Phân tích kỹ thuật phiên chiều 04/12: Tiếp diễ...
2,14/11/2024,BMP,https://vietstock.vn/2024/11/theo-dau-dong-tie...,Theo dấu dòng tiền cá mập 14/11: Khối ngoại ti...
3,22/10/2024,BMP,https://vietstock.vn/2024/10/nhua-binh-minh-sa...,Nhựa Bình Minh sắp chi gần 500 tỷ tạm ứng cổ t...
4,18/10/2024,BMP,https://vietstock.vn/2024/10/bmp-nghi-quyet-hd...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...
...,...,...,...,...
1303,11/02/2014,BSI,https://vietstock.vn/2014/02/chung-khoan-bsi-t...,Chứng khoán BSI: Thị trường tháng 2 sẽ dao độn...
1304,08/02/2014,BSI,https://vietstock.vn/2014/02/bsi-bao-cao-tinh-...,BSI: Báo cáo tình hình quản trị công ty 2013
1305,17/01/2014,BSI,https://vietstock.vn/2014/01/bsi-ttck-2014-tan...,BSI: TTCK 2014 tăng điểm chậm nhưng chắc
1306,16/01/2014,BSI,https://vietstock.vn/2014/01/bsi-bao-cao-tinh-...,BSI: Báo cáo tình hình quản trị công ty 2013


In [ ]:
# đặt BASE_DOMAIN phù hợp ở đây (đoán điển hình là Vietstock).
BASE_DOMAIN = "https://vietstock.vn"   # nếu link đã là tuyệt đối (http...), dòng này không ảnh hưởng

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123 Safari/537.36"
})

def normalize_url(url: str) -> str:
    """Chuẩn hóa URL: nhận tuyệt đối, thêm domain nếu là tương đối."""
    if not isinstance(url, str):
        return ""
    u = url.strip()
    if not u:
        return ""
    if u.startswith("http://") or u.startswith("https://"):
        return u
    if u.startswith("//"):
        return "https:" + u
    if u.startswith("/"):
        return urljoin(BASE_DOMAIN, u)
    # trường hợp như "vie/abc..." hoặc "vietstock.vn/..."
    if re.match(r"^[a-zA-Z0-9]+/.*", u):
        return urljoin(BASE_DOMAIN, "/" + u)
    if u.startswith("www."):
        return "https://" + u
    return u  # để nguyên nếu không đoán được

def fetch_phead(url: str, max_retries: int = 3, timeout: int = 20) -> str:
    """Tải trang và trích đoạn <p class='pHead'>; trả về chuỗi rỗng nếu thất bại."""
    u = normalize_url(url)
    if not u:
        return ""

    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(u, timeout=timeout, allow_redirects=True)
            # Một số trang chặn non-200 bằng JS; vẫn thử parse nếu có HTML
            if resp.status_code >= 500:
                raise requests.HTTPError(f"Server error {resp.status_code}")
            html = resp.text
            soup = BeautifulSoup(html, "lxml")

            # tìm <p class="pHead">
            node = soup.select_one("p.pHead")
            if node:
                text = node.get_text(" ", strip=True)
                if text:
                    return text

            # fallback: thử thêm các biến thể thường gặp
            fallback = soup.find("p", class_=re.compile(r"phead|pHead", re.I))
            if fallback:
                text = fallback.get_text(" ", strip=True)
                if text:
                    return text

            # thêm 1 fallback cuối: meta description
            meta = soup.find("meta", attrs={"name": "description"})
            if meta and meta.get("content"):
                return meta["content"].strip()

            return ""
        except Exception as e:
            # backoff nhẹ
            if attempt < max_retries:
                time.sleep(1.5 * attempt)
            else:
                return ""

# Đảm bảo có cột Link/Title
if "Link" not in df.columns or "Title" not in df.columns:
    raise ValueError("File Excel cần có cột 'Link' và 'Title'.")

# Crawl với progress bar
pheads = []
for url in tqdm(df["Link"].tolist(), desc="Crawling pHead"):
    pheads.append(fetch_phead(url))

df["pHead"] = pheads

# Tạo cột content = Title + ". " + pHead (nếu pHead rỗng vẫn tạo)
df["Title"] = df["Title"].fillna("").astype(str).str.strip()
df["pHead"] = df["pHead"].fillna("").astype(str).str.strip()
df["content"] = df.apply(lambda r: (r["Title"] + (". " if r["Title"] and r["pHead"] else ".") + r["pHead"]).strip(), axis=1)

df.head(10)

Crawling pHead: 100%|██████████| 1308/1308 [26:28<00:00,  1.21s/it]


,Date,Symbol,Link,Title,pHead,content
0,23/12/2024,BMP,https://vietstock.vn/2024/12/phan-tich-ky-thua...,Phân tích kỹ thuật phiên chiều 23/12: Tình hìn...,VN-Index và HNX-Index đồng loạt tăng điểm cùng...,Phân tích kỹ thuật phiên chiều 23/12: Tình hìn...
1,04/12/2024,BMP,https://vietstock.vn/2024/12/phan-tich-ky-thua...,Phân tích kỹ thuật phiên chiều 04/12: Tiếp diễ...,VN-Index và HNX-Index tăng giảm trái chiều đồn...,Phân tích kỹ thuật phiên chiều 04/12: Tiếp diễ...
2,14/11/2024,BMP,https://vietstock.vn/2024/11/theo-dau-dong-tie...,Theo dấu dòng tiền cá mập 14/11: Khối ngoại ti...,Trong bối cảnh VN-Index chốt phiên 14/11 mất h...,Theo dấu dòng tiền cá mập 14/11: Khối ngoại ti...
3,22/10/2024,BMP,https://vietstock.vn/2024/10/nhua-binh-minh-sa...,Nhựa Bình Minh sắp chi gần 500 tỷ tạm ứng cổ t...,Với việc sở hữu số dư tiền nhàn rỗi kỷ lục gần...,Nhựa Bình Minh sắp chi gần 500 tỷ tạm ứng cổ t...
4,18/10/2024,BMP,https://vietstock.vn/2024/10/bmp-nghi-quyet-hd...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...
5,18/10/2024,BMP,https://vietstock.vn/2024/10/bmp-nghi-quyet-hd...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...,BMP: Nghị quyết HĐQT về việc tạm ứng cổ tức ti...
6,18/10/2024,BMP,https://vietstock.vn/2024/10/nhua-binh-minh-la...,"Nhựa Bình Minh lại tạo kỷ lục mới, giá cổ phiế...",Lãi gộp quý 3/2024 của CTCP Nhựa Bình Minh ( H...,"Nhựa Bình Minh lại tạo kỷ lục mới, giá cổ phiế..."
7,16/10/2024,BMP,https://vietstock.vn/2024/10/phan-tich-ky-thua...,Phân tích kỹ thuật phiên chiều 16/10: Tiếp diễ...,VN-Index và HNX-Index tăng giảm trái chiều cùn...,Phân tích kỹ thuật phiên chiều 16/10: Tiếp diễ...
8,16/10/2024,BMP,https://vietstock.vn/2024/10/bmp-bctc-quy-3-na...,BMP: BCTC quý 3 năm 2024,BMP: BCTC quý 3 năm 2024,BMP: BCTC quý 3 năm 2024. BMP: BCTC quý 3 năm ...
9,16/10/2024,BMP,https://vietstock.vn/2024/10/bmp-bctc-hop-nhat...,BMP: BCTC Hợp nhất quý 3 năm 2024,BMP: BCTC Hợp nhất quý 3 năm 2024,BMP: BCTC Hợp nhất quý 3 năm 2024. BMP: BCTC H...


In [ ]:
out_xlsx = "output_with_content.xlsx"
df.to_excel(out_xlsx, index=False)

# ***CAFEF***

In [ ]:
import pandas as pd
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from bs4 import BeautifulSoup

vn30_symbols = ["PDR"]
url_template = "https://cafef.vn/du-lieu/tin-doanh-nghiep/{symbol}/event.chn"

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

def is_date_in_range(date_obj):
    start_date = datetime(2025, 6, 30)
    end_date = datetime(2025, 1, 1)
    return start_date <= date_obj <= end_date

articles_list = []

# Thu thập danh sách bài báo từ tất cả các trang
for symbol in vn30_symbols:
    url = url_template.format(symbol=symbol.lower())
    print(f"\nĐang xử lý: {symbol} ({url})")
    try:
        driver.get(url)
        time.sleep(7)
        page_count = 1
        last_page_content = None
        while True:
            soup = BeautifulSoup(driver.page_source, "html.parser")
            news_container = soup.find("div", class_="tintucsukien")
            if not news_container:
                print(f"Không tìm thấy tintucsukien cho {symbol}")
                break
            events_div = news_container.find("div", id="divEvents")
            if not events_div:
                print(f"Không tìm thấy divEvents cho {symbol}")
                break
            articles = events_div.find_all("li")
            print(f"Trang {page_count}: {len(articles)} bài báo")
            if len(articles) == 0:
                print(f"Không còn bài báo trên trang {page_count} cho {symbol}")
                break
            for article in articles:
                title_tag = article.find("a", class_="docnhanhTitle")
                if not title_tag:
                    continue
                title = title_tag.get_text(strip=True)
                link = "https://cafef.vn" + title_tag["href"] if not title_tag["href"].startswith("http") else title_tag["href"]
                date_tag = article.find("span", class_="timeTitle")
                if not date_tag:
                    continue
                date_str = date_tag.get_text(strip=True)
                try:
                    datetime_obj = datetime.strptime(date_str, "%d/%m/%Y %H:%M")
                    articles_list.append({"Symbol": symbol, "Title": title, "Link": link, "Date": datetime_obj})
                except ValueError as e:
                    print(f"Trang {page_count}: Lỗi định dạng ngày {date_str} - {e}")
                    continue
            current_page_content = driver.page_source
            try:
                next_button = driver.find_element(By.ID, "aNext")
                driver.execute_script("LoadNext();")
                time.sleep(2)
                new_page_content = driver.page_source
                if new_page_content == current_page_content or new_page_content == last_page_content:
                    print(f"Không có nội dung mới trên trang {page_count + 1} cho {symbol}")
                    break
                last_page_content = current_page_content
                page_count += 1
            except (NoSuchElementException, TimeoutException):
                print(f"Đã hết trang cho {symbol}")
                break
    except Exception as e:
        print(f"Lỗi khi xử lý {symbol}: {e}")
        continue

# Xử lý chi tiết từng bài báo (chỉ lấy tóm tắt)
data = []
for item in articles_list:
    try:
        if is_date_in_range(item["Date"]):
            driver.get(item["Link"])
            time.sleep(3)
            detail_soup = BeautifulSoup(driver.page_source, "html.parser")
            # Lấy tóm tắt từ h2 class="sapo"
            summary_h2 = detail_soup.find("h2", class_="sapo")
            content = summary_h2.get_text(strip=True) if summary_h2 else "N/A"
            print(f"Thêm {item['Title']} (ngày: {item['Date']}, tóm tắt: {content[:50]}...)")
            data.append({
                "Date": item["Date"],
                "Symbol": item["Symbol"],
                "Title": item["Title"],
                "Link": item["Link"],
                "Content": content
            })
    except Exception as e:
        print(f"Lỗi khi xử lý {item['Title']}: {e}")
        continue

driver.quit()

df = pd.DataFrame(data, columns=["Date", "Symbol", "Title", "Link", "Content"])
output_file = "cafef_vn30_news.xlsx"
df.to_excel(output_file, index=False, engine="openpyxl")
print(f"\nDữ liệu lưu tại {output_file}")
print(f"Tổng cộng: {len(data)} bài báo")
if df.empty:
    print("Cảnh báo: File trống, kiểm tra log")
else:
    print("DataFrame mẫu:")
    print(df.head())


Đang xử lý: PDR (https://cafef.vn/du-lieu/tin-doanh-nghiep/pdr/event.chn)
Trang 1: 30 bài báo
Trang 2: 30 bài báo
Trang 3: 30 bài báo
Trang 4: 30 bài báo
Trang 5: 30 bài báo
Trang 6: 30 bài báo
Trang 7: 30 bài báo
Trang 8: 30 bài báo
Trang 9: 30 bài báo
Trang 10: 30 bài báo
Trang 11: 30 bài báo
Trang 12: 30 bài báo
Trang 13: 30 bài báo
Trang 14: 30 bài báo
Trang 15: 30 bài báo
Trang 16: 30 bài báo
Trang 17: 30 bài báo
Trang 18: 30 bài báo
Trang 19: 30 bài báo
Trang 20: 30 bài báo
Trang 21: 30 bài báo
Trang 22: 30 bài báo
Trang 23: 30 bài báo
Trang 24: 30 bài báo
Trang 25: 30 bài báo
Trang 26: 30 bài báo
Trang 27: 30 bài báo
Trang 28: 30 bài báo
Trang 29: 30 bài báo
Trang 30: 30 bài báo
Trang 31: 30 bài báo
Trang 32: 30 bài báo
Trang 33: 30 bài báo
Trang 34: 30 bài báo
Trang 35: 30 bài báo
Trang 36: 30 bài báo
Trang 37: 30 bài báo
Trang 38: 30 bài báo
Trang 39: 30 bài báo
Trang 40: 30 bài báo
Trang 41: 30 bài báo
Trang 42: 30 bài báo
Trang 43: 30 bài báo
Trang 44: 30 bài báo
Trang 45: 

In [ ]:
vn30_symbols = ["PDR"]
url_template = "https://cafef.vn/du-lieu/tin-doanh-nghiep/{symbol}/event.chn"

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

def is_date_in_range(date_obj):
    """Kiểm tra ngày có trong khoảng từ 1/1/2025 đến 30/6/2025"""
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2025, 6, 30)
    return start_date <= date_obj <= end_date

def parse_date_flexible(date_str):
    """Xử lý parsing ngày với nhiều định dạng khác nhau"""
    # Loại bỏ khoảng trắng thừa
    date_str = date_str.strip()

    # Các định dạng có thể có
    formats = [
        "%d/%m/%Y %H:%M",
        "%d/%m/%Y",
        "%d.%m.%Y",
        "%d-%m-%Y %H:%M",
        "%d-%m-%Y"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue

    # Nếu không parse được, thử extract số
    try:
        # Tìm pattern dd/mm/yyyy hoặc dd.mm.yyyy
        date_match = re.search(r'(\d{1,2})[./](\d{1,2})[./](\d{4})', date_str)
        if date_match:
            day, month, year = map(int, date_match.groups())

            # Validate ngày tháng
            if month > 12:
                day, month = month, day  # Đổi chỗ nếu tháng > 12

            # Kiểm tra ngày hợp lệ
            try:
                return datetime(year, month, day)
            except ValueError:
                # Nếu ngày không hợp lệ, thử điều chỉnh
                if day > 31:
                    day = 31 if month in [1,3,5,7,8,10,12] else 30
                elif day > 30 and month in [4,6,9,11]:
                    day = 30
                elif day > 28 and month == 2:
                    day = 29 if year % 4 == 0 else 28

                return datetime(year, month, day)
    except:
        pass

    raise ValueError(f"Không thể parse ngày: {date_str}")

articles_list = []

# Thu thập danh sách bài báo từ tất cả các trang
for symbol in vn30_symbols:
    url = url_template.format(symbol=symbol.lower())
    print(f"\nĐang xử lý: {symbol} ({url})")
    try:
        driver.get(url)
        time.sleep(7)
        page_count = 1
        last_page_content = None

        while True:
            soup = BeautifulSoup(driver.page_source, "html.parser")
            news_container = soup.find("div", class_="tintucsukien")
            if not news_container:
                print(f"Không tìm thấy tintucsukien cho {symbol}")
                break

            events_div = news_container.find("div", id="divEvents")
            if not events_div:
                print(f"Không tìm thấy divEvents cho {symbol}")
                break

            articles = events_div.find_all("li")
            print(f"Trang {page_count}: {len(articles)} bài báo")

            if len(articles) == 0:
                print(f"Không còn bài báo trên trang {page_count} cho {symbol}")
                break

            for article in articles:
                title_tag = article.find("a", class_="docnhanhTitle")
                if not title_tag:
                    continue

                title = title_tag.get_text(strip=True)
                link = "https://cafef.vn" + title_tag["href"] if not title_tag["href"].startswith("http") else title_tag["href"]

                date_tag = article.find("span", class_="timeTitle")
                if not date_tag:
                    continue

                date_str = date_tag.get_text(strip=True)
                try:
                    datetime_obj = parse_date_flexible(date_str)
                    articles_list.append({
                        "Symbol": symbol,
                        "Title": title,
                        "Link": link,
                        "Date": datetime_obj
                    })
                except ValueError as e:
                    print(f"Trang {page_count}: Lỗi định dạng ngày '{date_str}' - {e}")
                    continue

            current_page_content = driver.page_source
            try:
                next_button = driver.find_element(By.ID, "aNext")
                driver.execute_script("LoadNext();")
                time.sleep(2)
                new_page_content = driver.page_source

                if new_page_content == current_page_content or new_page_content == last_page_content:
                    print(f"Không có nội dung mới trên trang {page_count + 1} cho {symbol}")
                    break

                last_page_content = current_page_content
                page_count += 1

            except (NoSuchElementException, TimeoutException):
                print(f"Đã hết trang cho {symbol}")
                break

    except Exception as e:
        print(f"Lỗi khi xử lý {symbol}: {e}")
        continue

print(f"\nTổng số bài báo thu thập: {len(articles_list)}")

# Xử lý chi tiết từng bài báo (chỉ lấy tóm tắt)
data = []
processed_count = 0

for item in articles_list:
    try:
        if is_date_in_range(item["Date"]):
            print(f"Đang xử lý: {item['Title']} ({item['Date'].strftime('%d/%m/%Y')})")
            driver.get(item["Link"])
            time.sleep(3)

            detail_soup = BeautifulSoup(driver.page_source, "html.parser")

            # Lấy tóm tắt từ h2 class="sapo"
            summary_h2 = detail_soup.find("h2", class_="sapo")
            content = summary_h2.get_text(strip=True) if summary_h2 else "N/A"

            data.append({
                "Date": item["Date"],
                "Symbol": item["Symbol"],
                "Title": item["Title"],
                "Link": item["Link"],
                "Content": content
            })

            processed_count += 1
            print(f"✓ Đã xử lý ({processed_count}/{len([x for x in articles_list if is_date_in_range(x['Date'])])})")
        else:
            print(f"Bỏ qua bài ngoài khoảng thời gian: {item['Title']} ({item['Date'].strftime('%d/%m/%Y')})")

    except Exception as e:
        print(f"Lỗi khi xử lý '{item['Title']}': {e}")
        continue

driver.quit()

# Lưu dữ liệu
df = pd.DataFrame(data, columns=["Date", "Symbol", "Title", "Link", "Content"])
output_file = "cafef_vn30_news.xlsx"

try:
    df.to_excel(output_file, index=False, engine="openpyxl")
    print(f"\n✓ Dữ liệu đã lưu tại {output_file}")
    print(f"✓ Tổng cộng: {len(data)} bài báo trong khoảng thời gian 01/01/2025 - 30/06/2025")

    if not df.empty:
        print("\nDataFrame mẫu:")
        print(df.head())
        print(f"\nThống kê theo ngày:")
        print(df['Date'].dt.strftime('%Y-%m').value_counts().sort_index())
    else:
        print("⚠️ Cảnh báo: File trống, kiểm tra khoảng thời gian và dữ liệu")

except Exception as e:
    print(f"Lỗi khi lưu file: {e}")


Đang xử lý: PDR (https://cafef.vn/du-lieu/tin-doanh-nghiep/pdr/event.chn)
Trang 1: 30 bài báo
Trang 2: 30 bài báo
Trang 3: 30 bài báo
Trang 4: 30 bài báo
Trang 5: 30 bài báo
Trang 6: 30 bài báo
Trang 7: 30 bài báo
Trang 8: 30 bài báo
Trang 9: 30 bài báo
Trang 10: 30 bài báo
Trang 11: 30 bài báo
Trang 12: 30 bài báo
Trang 13: 30 bài báo
Trang 14: 30 bài báo
Trang 15: 30 bài báo
Trang 16: 30 bài báo
Trang 17: 30 bài báo
Trang 18: 30 bài báo
Trang 19: 30 bài báo
Trang 20: 30 bài báo
Trang 21: 30 bài báo
Trang 22: 30 bài báo
Trang 23: 30 bài báo
Trang 24: 30 bài báo
Trang 25: 30 bài báo
Trang 26: 30 bài báo
Trang 27: 30 bài báo
Trang 28: 30 bài báo
Trang 29: 30 bài báo
Trang 30: 30 bài báo
Trang 31: 30 bài báo
Trang 32: 30 bài báo
Trang 33: 30 bài báo
Trang 34: 30 bài báo
Trang 35: 30 bài báo
Trang 36: 30 bài báo
Trang 37: 30 bài báo
Trang 38: 30 bài báo
Trang 39: 30 bài báo
Trang 40: 30 bài báo
Trang 41: 30 bài báo
Trang 42: 30 bài báo
Trang 43: 30 bài báo
Trang 44: 24 bài báo
Trang 45: 